# Assignment 2 · Diagnose and improve generated images
**GAN track · Notebook v9 · Chonnam National University · Fall 2026**

Choose **VAE OR GAN**, not both. / VAE 또는 GAN 중 **하나만** 선택하세요. Released 16 October 2026 · Due 4 November 2026, 23:59 KST · 12 points · 10% of course grade.

Inspect generated MNIST images, compare A–C, then choose **one extra setting** for Q4. All runs restart from the same initial seed and use the same fixed `z` panel for comparison. Improved images are **not required** for full credit. / 생성 이미지의 문제를 찾고 변경 계획을 먼저 쓴 뒤 실험하세요. Q4에서는 설정 하나를 더 시험합니다. 결과가 개선되지 않아도 근거 있는 분석이면 만점을 받을 수 있습니다.

GAN: three runs compare baseline G learning rate 0.00004, higher G learning rate 0.0002, then G update frequency at that higher rate. D learning rate stays 0.0002. All use non-saturating G loss.

Active work target: **3–5 hours** including a 16-image audit per run and 1–2-page analysis. Four bounded CPU runs; no broad hyperparameter search. Submit the executed notebook and **one PDF** with your 1–2-page analysis followed by the three-page receipt. See `A2_Student_Assignment_v10.pdf`. English or Korean answers are accepted; technical terms can remain in English.

## 0 Setup and data / 준비
Run these supplied cells. `auto` downloads MNIST if needed. If download fails, an offline sklearn digits dataset runs as a smoke test, but its result is **not submission-ready**. No Drive mount, GPU, or paid service is required. Reference code is openly provided through `check_code(N, use_reference=True)` and has **no penalty**; explain each block in your report.

In [ ]:
#@title Setup and workflow / 준비 및 실행 관리 {display-mode:"form"}
import copy, gzip, hashlib, json, math, os, platform, struct, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, FileLink

torch.set_num_threads(2)
VERSION = 'A2 redesign v10 / notebook v9'
SEED = 0
BASE, VARIANT, SECOND, PLAN, AUDIT = None, None, None, None, None
EXTRA, EXTRA_PLAN, EXTRA_AUDIT = None, None, None
CODE_STATUS = {}
OUT = Path('a2_outputs'); OUT.mkdir(exist_ok=True)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:16]

def array_id(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

def code_id():
    # Detect ordinary edits after a run; this is not an anti-tampering system.
    return fingerprint({name: (globals()[name].__code__.co_code.hex(), repr(globals()[name].__code__.co_consts)) for name in CODE_NAMES})

def check_code(number, use_reference=False):
    name = CODE_NAMES[number - 1]
    if use_reference:
        globals()[name] = REFERENCES[number - 1]
    try:
        test_student_code(number)
        mode = 'reference' if globals()[name] is REFERENCES[number - 1] else 'student'
        CODE_STATUS[name] = mode
        print(f'Code {number}: PASS ({mode})')
        return True
    except Exception as exc:
        CODE_STATUS[name] = 'pending'
        print(f'Code {number}: NOT READY. {type(exc).__name__}: {exc}')
        print('update, tensor shape, eval/no_grad를 확인하세요. 필요하면 check_code(number, use_reference=True)를 사용하세요. 감점은 없습니다.')
        return False

def record_plan(student_id, diagnosis, candidate, prediction, success_rule, second_prediction=None, g_updates=None):
    global PLAN, VARIANT, SECOND, AUDIT, EXTRA, EXTRA_PLAN, EXTRA_AUDIT
    if BASE is None:
        print('먼저 baseline을 실행하세요 / Run the baseline first.'); return
    values = [student_id, diagnosis, prediction, success_rule]
    if any(not isinstance(x, str) or not x.strip() for x in values):
        print('NOT READY: 학번, baseline 관찰, 예측, 판단 기준을 작성하세요.'); return
    if not isinstance(second_prediction,str) or not second_prediction.strip():
        print('NOT READY: 두 번째 변경에 대한 예측도 작성하세요.'); return
    if candidate not in OPTIONS:
        print('NOT READY: 이 track에서 허용된 변경값을 사용하세요 / use the assigned option.'); return
    if TRACK == 'GAN' and (type(g_updates) is not int or g_updates != G_UPDATES_SECOND):
        print(f'NOT READY: C run의 G updates 값을 {G_UPDATES_SECOND}로 입력하세요.'); return
    draft = dict(student_id=student_id.strip(), diagnosis=diagnosis.strip(), candidate=float(candidate),
                 prediction=prediction.strip(), success_rule=success_rule.strip(), baseline_id=BASE['id'])
    draft['second_prediction']=second_prediction.strip()
    if TRACK == 'GAN': draft['g_updates']=g_updates
    if PLAN != draft:
        PLAN = draft; VARIANT = SECOND = AUDIT = None
        EXTRA = EXTRA_PLAN = EXTRA_AUDIT = None
    print('PLAN SAVED. 변경 실험을 실행하세요 / Run your chosen intervention.')

def run_baseline():
    global BASE, VARIANT, SECOND, PLAN, AUDIT, EXTRA, EXTRA_PLAN, EXTRA_AUDIT
    cfg = config(BASE_VALUE)
    if BASE is not None and BASE['config'] == cfg:
        print('기존 baseline 재사용 / Reusing baseline.'); return
    BASE = train(BASE_VALUE, student=False)
    PLAN = VARIANT = SECOND = AUDIT = None
    EXTRA = EXTRA_PLAN = EXTRA_AUDIT = None

def run_variant():
    global VARIANT, SECOND, AUDIT, EXTRA, EXTRA_PLAN, EXTRA_AUDIT
    if PLAN is None or BASE is None:
        print('NOT READY: baseline을 보고 변경 계획을 먼저 작성하세요.'); return
    if BASE['config'] != config(BASE_VALUE) or PLAN['baseline_id'] != BASE['id']:
        print('NOT READY: 설정이 바뀌었습니다. baseline부터 다시 실행하세요.'); return
    if not all([check_code(i) for i in [1, 2, 3]]):
        print('NOT READY: 코드 세 칸을 완성하거나 제공 코드로 전환하세요.'); return
    key = fingerprint(PLAN)
    if VARIANT is not None and SECOND is not None and VARIANT['plan_id'] == key and VARIANT['code_id'] == code_id():
        print('기존 변경 실험 재사용 / Reusing intervention run.'); return
    VARIANT = train(PLAN['candidate'], student=True)
    VARIANT.update(plan_id=key, code_id=code_id())
    SECOND = train(PLAN['candidate'], student=True, g_updates=PLAN['g_updates']) if TRACK=='GAN' else train(SECOND_VALUE,student=True)
    SECOND.update(plan_id=key, code_id=code_id())
    AUDIT = None
    EXTRA = EXTRA_PLAN = EXTRA_AUDIT = None

def ready_reason(include_extra=False):
    if BASE is None or PLAN is None or VARIANT is None:
        return 'baseline, plan 또는 intervention 실행이 남아 있습니다.'
    if BASE['config'] != config(BASE_VALUE) or VARIANT['config'] != config(PLAN['candidate']):
        return '설정이 바뀌었습니다. baseline부터 다시 실행하세요.'
    if VARIANT['plan_id'] != fingerprint(PLAN) or PLAN['baseline_id'] != BASE['id']:
        return '변경 계획과 실행 결과가 다릅니다.'
    if VARIANT['code_id'] != code_id() or any(CODE_STATUS.get(n) == 'pending' for n in CODE_NAMES):
        return '코드가 바뀌었습니다. 코드 점검과 변경 실험을 다시 실행하세요.'
    expected_second=config(PLAN['candidate'],PLAN['g_updates']) if TRACK=='GAN' else config(SECOND_VALUE)
    if (SECOND is None or SECOND['config'] != expected_second
        or SECOND['plan_id'] != fingerprint(PLAN) or SECOND['code_id'] != code_id()):
        return '두 번째 변경 실험이 남아 있거나 설정이 바뀌었습니다.'
    if include_extra:
        if EXTRA_PLAN is None or EXTRA is None or EXTRA_AUDIT is None:
            return 'Q4 추가 실험·16-image audit가 남아 있습니다.'
        if EXTRA_PLAN['core_id'] != fingerprint([BASE['id'],VARIANT['id'],SECOND['id']]):
            return 'Q4 계획 이전 결과가 바뀌었습니다. Q4를 다시 실행하세요.'
        if EXTRA['config'] != extra_config(EXTRA_PLAN) or EXTRA.get('extra_plan_id') != fingerprint(EXTRA_PLAN) or EXTRA.get('code_id') != code_id():
            return 'Q4 설정 또는 코드가 바뀌었습니다. 추가 실험을 다시 실행하세요.'
        if EXTRA_AUDIT['extra_id'] != EXTRA['id']:
            return 'Q4 image audit를 다시 작성하세요.'
    return ''


def record_extra_plan(factor, value, prediction, decision_rule):
    global EXTRA_PLAN, EXTRA, EXTRA_AUDIT
    reason=ready_reason()
    if reason:
        print('NOT READY:',reason);return
    if not isinstance(prediction,str) or not prediction.strip() or not isinstance(decision_rule,str) or not decision_rule.strip():
        print('NOT READY: Q4 예측과 비교 기준을 먼저 작성하세요.');return
    try:
        cfg=extra_config(dict(factor=factor,value=value))
    except (ValueError,TypeError) as exc:
        print('NOT READY:',exc);return
    draft=dict(factor=factor,value=value,prediction=prediction.strip(),decision_rule=decision_rule.strip(),
               core_id=fingerprint([BASE['id'],VARIANT['id'],SECOND['id']]),config=cfg)
    if EXTRA_PLAN!=draft:
        EXTRA_PLAN=draft;EXTRA=EXTRA_AUDIT=None
    print('Q4 PLAN SAVED. 추가 run을 실행하세요.')


def run_extra():
    global EXTRA, EXTRA_AUDIT
    if EXTRA_PLAN is None:
        print('NOT READY: Q4 설정과 예측을 먼저 기록하세요.');return
    reason=ready_reason()
    if reason:
        print('NOT READY:',reason);return
    key=fingerprint(EXTRA_PLAN)
    if EXTRA is not None and EXTRA.get('extra_plan_id')==key and EXTRA.get('code_id')==code_id():
        print('기존 Q4 run 재사용 / Reusing extra run.');return
    EXTRA=train_extra(EXTRA_PLAN)
    EXTRA.update(extra_plan_id=key,code_id=code_id())
    EXTRA_AUDIT=None


def save_extra_audit(labels_text):
    global EXTRA_AUDIT
    if EXTRA is None:
        print('NOT READY: Q4 run을 먼저 실행하세요.');return
    labels=[x.strip().lower() for x in labels_text.split(',')]
    if len(labels)!=16 or any(x not in list('0123456789')+['x'] for x in labels):
        print('NOT READY: Q4 sample 01–16을 0–9 또는 x로 입력하세요.');return
    EXTRA_AUDIT=dict(extra_id=EXTRA['id'],labels=labels)
    print('Q4 visual audit saved.')


def show_extra():
    if EXTRA is None:
        print('NOT READY: Q4 run을 먼저 실행하세요.');return
    fig=extra_figure();fig.savefig(OUT/f'{TRACK}_Q4_extra.png',dpi=130)
    plt.show();plt.close(fig)
    print('Q4 setting:',EXTRA['config'])
    print('Q4 measures:',metrics(EXTRA))

def make_receipt(ai_use):
    reason = ready_reason(include_extra=True)
    if not reason and 'OFFLINE' in DATA_NAME:
        reason = '오프라인 smoke test입니다. 제출 전 MNIST로 두 run을 실행하세요.'
    if not reason and (not isinstance(ai_use, str) or not ai_use.strip()):
        reason = 'AI 사용 내용을 쓰세요. 사용하지 않았다면 none을 적으세요.'
    if not reason and (AUDIT is None or AUDIT['run_id'] != VARIANT['id'] or AUDIT.get('second_id') != SECOND['id']):
        reason = '생성 grid의 sample 01–16에 대한 visual audit를 작성하세요.'
    if reason:
        print('NOT READY:', reason); return None
    b, v = metrics(BASE), metrics(VARIANT)
    payload = dict(version=VERSION, track=TRACK, plan=PLAN, code=CODE_STATUS.copy(),
                   baseline=dict(config=BASE['config'], metrics=b, sample_id=array_id(BASE['samples'])),
                   intervention=dict(config=VARIANT['config'], metrics=v, sample_id=array_id(VARIANT['samples'])),
                   visual_audit=AUDIT, ai_use=ai_use.strip(),
                   environment=dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, device='cpu'))
    payload['second_change']=dict(config=SECOND['config'],metrics=metrics(SECOND),sample_id=array_id(SECOND['samples']))
    payload['extra_change']=dict(plan=EXTRA_PLAN,config=EXTRA['config'],metrics=metrics(EXTRA),
                                 sample_id=array_id(EXTRA['samples']),visual_audit=EXTRA_AUDIT)
    fid=fingerprint(payload)
    receipt=dict(status='READY', fingerprint=fid, **payload)
    (OUT/f'A2_{TRACK}_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf8')
    # English labels and numbers in the PDF avoid missing Korean fonts on Colab.
    # The full original Korean plan/AI statement is preserved in the notebook and JSON.
    lines=[f'{VERSION} | {TRACK} | READY',f'Fingerprint: {fid}',
           f'Data: {BASE["config"]["data"]}', f'Changed parameter: {PARAMETER}',
           f'Baseline: {BASE_VALUE}   First change: {PLAN["candidate"]}   Second change: {SECOND["config"]}',
           'All runs restart from the same seed; comparisons change one factor at a time.',
           'Settings: '+json.dumps(BASE['config'],ensure_ascii=True), '',
           'BASELINE', *[f'  {k}: {v}' for k,v in b.items()], '',
           'INTERVENTION', *[f'  {k}: {v}' for k,v in v.items()], '',
           'SECOND CHANGE',*[f'  {k}: {v}' for k,v in metrics(SECOND).items()],'',
           'Q4 EXTRA CHANGE',f'  factor: {EXTRA_PLAN["factor"]}  value: {EXTRA_PLAN["value"]}',
           *[f'  {k}: {v}' for k,v in metrics(EXTRA).items()],'',
           'Code: '+str(CODE_STATUS),
           'Full plan, AI statement and annotations: notebook and receipt JSON.',
           'This receipt is an appendix. Write your own analysis in the report.',
           'Fingerprint checks consistency; it does not prove authorship.']
    import textwrap
    wrapped='\n'.join(y for x in lines for y in (textwrap.wrap(x,95) or ['']))
    pdf_path=OUT/f'A2_{TRACK}_receipt.pdf'
    with PdfPages(pdf_path) as pdf:
        fig=plt.figure(figsize=(8.27,11.69));fig.text(.07,.95,wrapped,va='top',fontfamily='monospace',fontsize=8.5);pdf.savefig(fig);plt.close(fig)
        fig=comparison_figure();pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
        fig=extra_figure();pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('READY | fingerprint:',fid)
    print('Baseline:',b); print('Intervention:',v)
    print('분석 보고서 뒤에 PDF를 붙이세요. JSON은 노트북과 함께 보관하면 됩니다.')
    display(FileLink(str(pdf_path)));display(FileLink(str(OUT/f'A2_{TRACK}_receipt.json')))
    return receipt

In [ ]:
#@title Model, data, training and figures / model, data, 학습 및 그림 {display-mode:"form"}
TRACK='GAN'
PARAMETER='generator learning rate, then G updates per D update'
BASE_VALUE=0.00004
OPTIONS=(0.0002,)
EPOCHS=3; BATCH=128; Z_DIM=16; D_LR=0.0002
G_UPDATES_SECOND=5
DATA_SOURCE='auto';DATA_NAME=None
Z_FIXED=torch.randn(64,Z_DIM,generator=torch.Generator().manual_seed(7))

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(Z_DIM,128*7*7),nn.BatchNorm1d(128*7*7),nn.ReLU())
        self.up=nn.Sequential(nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.ConvTranspose2d(64,1,4,2,1),nn.Tanh())
    def forward(self,z):return self.up(self.fc(z).view(-1,128,7,7))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,64,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(128*7*7,1))
    def forward(self,x):return self.net(x)

def reference_d_update(D,opt,real,fake):
    real_logits=D(real);fake_logits=D(fake.detach())
    loss=F.binary_cross_entropy_with_logits(real_logits,torch.ones_like(real_logits))+F.binary_cross_entropy_with_logits(fake_logits,torch.zeros_like(fake_logits))
    opt.zero_grad();loss.backward();opt.step()
    return float(loss.detach())

def reference_g_update(D,opt,fake):
    fake_logits=D(fake)
    loss=F.binary_cross_entropy_with_logits(fake_logits,torch.ones_like(fake_logits))
    opt.zero_grad();loss.backward();opt.step()
    return float(loss.detach())

def reference_generate_images(G,z):
    was_training=G.training;G.eval()
    with torch.no_grad(): out=((G(z)+1)/2).clamp(0,1).cpu().numpy()
    G.train(was_training)
    return out

CODE_NAMES=['d_update','g_update','generate_images']
REFERENCES=[reference_d_update,reference_g_update,reference_generate_images]

def test_student_code(number):
    fn=globals()[CODE_NAMES[number-1]]
    torch.manual_seed(709)
    if number==1:
        a=Discriminator();b=copy.deepcopy(a)
        oa=torch.optim.SGD(a.parameters(),lr=.001);ob=torch.optim.SGD(b.parameters(),lr=.001)
        real=torch.randn(3,1,28,28);fake=torch.randn(3,1,28,28,requires_grad=True)
        expected=reference_d_update(a,oa,real,fake)
        actual=fn(b,ob,real,fake)
        assert actual is not None and abs(actual-expected)<1e-5, 'D loss differs'
        assert fake.grad is None, 'D update must detach fake images'
        assert all(torch.allclose(x,y,atol=1e-5,rtol=1e-4) for x,y in zip(a.parameters(),b.parameters())), 'D optimizer update differs'
    elif number==2:
        g1=Generator();g2=copy.deepcopy(g1);d=Discriminator()
        o1=torch.optim.SGD(g1.parameters(),lr=.001);o2=torch.optim.SGD(g2.parameters(),lr=.001)
        z=torch.randn(3,Z_DIM)
        f1=g1(z);f2=g2(z)
        expected=reference_g_update(d,o1,f1)
        actual=fn(d,o2,f2)
        assert actual is not None and abs(actual-expected)<1e-5, 'G loss differs'
        assert all(torch.allclose(x,y,atol=1e-5,rtol=1e-4) for x,y in zip(g1.parameters(),g2.parameters())), 'G optimizer update differs; keep the gradient path through D'
    else:
        g=Generator();g.train();z=torch.randn(4,Z_DIM)
        expected=reference_generate_images(g,z)
        actual=fn(g,z)
        assert actual is not None and isinstance(actual,np.ndarray), 'return NumPy images'
        assert actual.shape==expected.shape and np.allclose(actual,expected,atol=1e-6), 'images differ; use eval mode and map [-1,1] to [0,1]'
        assert g.training, 'restore the original training mode'

def load_data(source='auto'):
    global X_TRAIN,X_TEST,DATA_NAME,DATA_SOURCE
    DATA_SOURCE=source
    if source not in ['auto','offline']:raise ValueError('source must be auto or offline')
    try:
        if source=='offline':raise OSError('offline mode requested')
        arrays=[]
        for name in ['train-images-idx3-ubyte.gz','t10k-images-idx3-ubyte.gz']:
            p=Path(name)
            if not p.exists():
                last=None
                for base in ['https://ossci-datasets.s3.amazonaws.com/mnist/','https://storage.googleapis.com/cvdf-datasets/mnist/']:
                    try:
                        with urllib.request.urlopen(base+name,timeout=8) as response:raw=response.read()
                        decoded=gzip.decompress(raw);magic,n,r,c=struct.unpack('>IIII',decoded[:16])
                        assert magic==2051 and r==28 and c==28 and len(decoded)==16+n*r*c
                        p.write_bytes(raw);break
                    except Exception as exc:last=exc
                else:raise OSError(f'MNIST download failed: {last}')
            with gzip.open(p,'rb') as f:
                magic,n,r,c=struct.unpack('>IIII',f.read(16));assert magic==2051 and r*c==784
                arrays.append(torch.from_numpy(np.frombuffer(f.read(),np.uint8).copy().reshape(n,1,28,28)).float()/127.5-1)
        X_TRAIN,X_TEST=arrays[0][:20000],arrays[1]
        DATA_NAME='MNIST first 20000 train / 10000 test'
    except (OSError,AssertionError,ValueError,EOFError,struct.error) as exc:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        ds=load_digits();x=torch.tensor(ds.images/16,dtype=torch.float32)[:,None]
        x=F.interpolate(x,size=(28,28),mode='bilinear',align_corners=False)*2-1
        tr,te=train_test_split(np.arange(len(x)),test_size=.2,random_state=42,stratify=ds.target)
        X_TRAIN,X_TEST=x[tr],x[te];DATA_NAME='OFFLINE sklearn digits SMOKE TEST ONLY'
        print('MNIST unavailable:',type(exc).__name__)
        print('오프라인 실행 점검용 data입니다. 제출용 MNIST와 다르며 READY PDF를 만들 수 없습니다.')
    print(DATA_NAME,tuple(X_TRAIN.shape),tuple(X_TEST.shape))

def config(value,g_updates=1):
    return dict(data=DATA_NAME,seed=SEED,epochs=EPOCHS,batch=BATCH,z_dim=Z_DIM,g_lr=float(value),d_lr=D_LR,
                train_size=len(X_TRAIN),loss='non-saturating',optimizer='Adam betas=(0.5,0.999)',
                updates=f'D1 G{g_updates}',fixed_z_seed=7)

def train(value,student=False,g_updates=1):
    start=time.time();torch.manual_seed(SEED)
    G,D=Generator(),Discriminator()
    og=torch.optim.Adam(G.parameters(),lr=value,betas=(.5,.999))
    od=torch.optim.Adam(D.parameters(),lr=D_LR,betas=(.5,.999))
    history=[];snaps={}
    for ep in range(1,EPOCHS+1):
        perm=torch.randperm(len(X_TRAIN));ld=lg=dr=df=0.;n=0
        for i in range(0,len(X_TRAIN),BATCH):
            real=X_TRAIN[perm[i:i+BATCH]]
            fake=G(torch.randn(len(real),Z_DIM))
            dloss=(d_update if student else reference_d_update)(D,od,real,fake)
            gloss=(g_update if student else reference_g_update)(D,og,fake)
            for _ in range(1,g_updates):
                extra_fake=G(torch.randn(len(real),Z_DIM))
                gloss=(g_update if student else reference_g_update)(D,og,extra_fake)
            with torch.no_grad():
                dr+=float(D(real).sigmoid().mean());df+=float(D(fake.detach()).sigmoid().mean())
            ld+=dloss;lg+=gloss;n+=1
        snaps[ep]=(generate_images if student else reference_generate_images)(G,Z_FIXED)
        history.append([ep,ld/n,lg/n,dr/n,df/n])
        print(f'epoch {ep}/{EPOCHS} | D loss {ld/n:.3f} | G loss {lg/n:.3f} | D(real) {dr/n:.3f} | D(fake) {df/n:.3f}')
    samples=snaps[EPOCHS]
    assert np.isfinite(samples).all()
    result=dict(config=config(value,g_updates),samples=samples,snapshots=snaps,history=history,seconds=round(time.time()-start,2))
    result['id']=fingerprint([result['config'],array_id(samples)])
    print('Completed in',result['seconds'],'s. G lr =',value,'G updates =',g_updates)
    return result


def extra_config(plan):
    factor,value=plan['factor'],plan['value']
    if factor=='g_lr' and type(value) in (int,float) and float(value)==0.0004:
        return config(0.0004,1)
    if factor=='g_updates' and type(value) is int and value==3:
        return config(0.0002,3)
    raise ValueError('GAN Q4: choose g_lr=0.0004 or g_updates=3; change one factor only.')


def train_extra(plan):
    extra_config(plan)
    return train(0.0004,student=True,g_updates=1) if plan['factor']=='g_lr' else train(0.0002,student=True,g_updates=3)

def _grid(fig,slot,images,rows=4,title=None,label_all=False):
    if title:
        heading=fig.add_subplot(slot);heading.set_title(title,fontsize=11,pad=8);heading.axis('off')
    sub=slot.subgridspec(rows,8,wspace=.02,hspace=.12)
    for i in range(rows*8):
        ax=fig.add_subplot(sub[i//8,i%8]);ax.imshow(images[i].reshape(28,28),cmap='gray',vmin=0,vmax=1);ax.axis('off')
        if i<16 or label_all:ax.set_title(str(i+1).zfill(2),fontsize=6 if label_all else 7,pad=0)

def baseline_figure():
    if BASE is None:return
    fig=plt.figure(figsize=(10,5));gs=fig.add_gridspec(1,2)
    _grid(fig,gs[0],((X_TEST[:32]+1)/2).numpy())
    _grid(fig,gs[1],BASE['samples'])
    fig.text(.26,.96,'Real MNIST',ha='center');fig.text(.75,.96,'Baseline generated | fixed z',ha='center')
    plt.show();plt.close(fig)

def comparison_figure():
    fig=plt.figure(figsize=(14,10))
    gs=fig.add_gridspec(4,3,left=.035,right=.98,bottom=.04,top=.91,hspace=.42,wspace=.09)
    _grid(fig,gs[0,0],((X_TEST[:16]+1)/2).numpy(),rows=2,title='Real MNIST · reference only')
    guide=fig.add_subplot(gs[0,1:]);guide.axis('off')
    guide.text(.02,.82,'How to read',fontsize=12,fontweight='bold',va='top')
    guide.text(.02,.60,'Rows: epoch 1 → 2 → 3    |    Columns: baseline → higher G learning rate → more G updates\n'
                    'Same generated ID = same fixed z across all runs and epochs; the real row is not paired.\n'
                    'Audit final epoch IDs 01–16 in all three generated columns.',fontsize=10,va='top',linespacing=1.4)
    runs=[(BASE,'Baseline · G lr 0.00004 · D1:G1'),
          (VARIANT,'LR change · G lr 0.0002 · D1:G1'),
          (SECOND,f'Balance change · G lr 0.0002 · D1:G{G_UPDATES_SECOND}')]
    for epoch in [1,2,3]:
        for column,(run,label) in enumerate(runs):
            _grid(fig,gs[epoch,column],run['snapshots'][epoch][:16],rows=2,title=f'{label} · epoch {epoch}')
    fig.suptitle('GAN image formation from fixed z · read each comparison left to right',y=.98,fontsize=14)
    return fig


def comparison_64_figure():
    fig=plt.figure(figsize=(18,7))
    gs=fig.add_gridspec(1,3,left=.03,right=.99,bottom=.04,top=.85,wspace=.08)
    for column,(run,title) in enumerate([
        (BASE,'A baseline'),(VARIANT,'B higher G learning rate'),
        (SECOND,f'C D1:G{G_UPDATES_SECOND}')]):
        _grid(fig,gs[column],run['samples'],rows=8,title=title,label_all=True)
    fig.suptitle('Fixed-z sample check · IDs 01–64 · same z across A, B and C',y=.98,fontsize=14)
    return fig


def extra_figure():
    if EXTRA_PLAN['factor']=='g_lr':
        runs=[(BASE,'A · G lr 0.00004'),(VARIANT,'B · G lr 0.0002'),(EXTRA,'Q4 · G lr 0.0004')]
    else:
        runs=[(VARIANT,'B · D1:G1'),(EXTRA,'Q4 · D1:G3'),(SECOND,'C · D1:G5')]
    fig=plt.figure(figsize=(15,7));gs=fig.add_gridspec(1,3,left=.03,right=.99,bottom=.04,top=.86,wspace=.09)
    for col,(run,title) in enumerate(runs):_grid(fig,gs[col],run['samples'][:32],rows=4,title=title)
    fig.suptitle('Q4 · same fixed z · compare 32 generated images per setting',fontsize=14,y=.98)
    return fig

def save_audit(baseline_labels,intervention_labels,balance_labels):
    global AUDIT
    if ready_reason():print('NOT READY: 먼저 변경 실험을 완료하세요.');return
    parsed=[]
    for item in [baseline_labels,intervention_labels,balance_labels]:
        labels=[x.strip().lower() for x in item.split(',')]
        if len(labels)!=16 or any(x not in list('0123456789')+['x'] for x in labels):
            AUDIT=None;print('NOT READY: 01–16 labels must be 0–9 or x, separated by commas.');return
        parsed.append(labels)
    AUDIT=dict(run_id=VARIANT['id'],second_id=SECOND['id'],baseline=parsed[0],intervention=parsed[1],second=parsed[2])
    print('Three visual audits saved. These 16 subjective labels are not classifier accuracy or proof of mode collapse.')

def metrics(run):
    h=run['history'][-1]
    out=dict(final_D_loss=round(h[1],3),final_G_loss=round(h[2],3),D_real=round(h[3],3),D_fake=round(h[4],3))
    if run is EXTRA:
        if EXTRA_AUDIT is not None and EXTRA_AUDIT['extra_id']==EXTRA['id']:
            labels=EXTRA_AUDIT['labels']
            out.update(audited_samples=16,readable_share=round(float(np.mean(np.asarray(labels)!='x')),4),
                       distinct_readable_digits=len(set(labels)-{'x'}),labels=labels)
        return out
    if AUDIT is not None and VARIANT is not None and SECOND is not None and AUDIT['run_id']==VARIANT['id'] and AUDIT['second_id']==SECOND['id']:
        labels=AUDIT['baseline' if run is BASE else 'intervention' if run is VARIANT else 'second']
        out.update(audited_samples=16,readable_share=round(float(np.mean(np.asarray(labels)!='x')),4),
                   distinct_readable_digits=len(set(labels)-{'x'}),labels=labels)
    return out

def show_comparison():
    reason=ready_reason()
    if reason:print('NOT READY:',reason);return
    fig=comparison_figure();fig.savefig(OUT/'GAN_comparison.png',dpi=120);plt.show();plt.close(fig)
    fig=comparison_64_figure();fig.savefig(OUT/'GAN_64samples.png',dpi=120);plt.show();plt.close(fig)
    print('Baseline',metrics(BASE));print('LR change',metrics(VARIANT));print('Balance change',metrics(SECOND))
    print('Audit final epoch 3, IDs 01–16 in all three columns. Use x when unclear. Inspect IDs 17–64 for additional repeated or distinct patterns; no extra labels are required.')

def optional_loss_plot():
    if ready_reason():return
    fig,axes=plt.subplots(1,4,figsize=(12,3))
    for run,label in [(BASE,'Baseline'),(VARIANT,'LR change'),(SECOND,'Balance change')]:
        h=np.array(run['history'])
        for j in range(4):axes[j].plot(h[:,0],h[:,j+1],marker='o',label=label)
    for a,t in zip(axes,['D loss','G loss','D(real)','D(fake)']):a.set(xlabel='Epoch',title=t);a.legend(fontsize=7)
    fig.tight_layout();plt.show();plt.close(fig)

In [ ]:
#@title MNIST data / MNIST 준비
load_data('auto')
# To check execution without internet: load_data('offline'). Restore auto for submission.

## 1 Observe baseline images / Baseline 관찰
Run the baseline and compare **real MNIST** with **prior/generated** images. Name a visible limitation using sample IDs 01–16. Do not diagnose a learning mechanism from loss alone. / 먼저 실제 이미지와 생성 이미지를 보고 sample 번호로 문제를 짚으세요.

In [ ]:
#@title Baseline training and samples / Baseline 학습과 생성 이미지
run_baseline()
baseline_figure()

## 2 Record the intervention plan / 변경 전 계획
Compare three fresh runs: A baseline (G learning rate 0.00004; D1:G1), B learning-rate change (G learning rate 0.0002; D1:G1), and C update-balance change (G learning rate 0.0002; D1:G5). Enter **both** B's G learning rate and C's G updates in the plan cell before training; do not change other settings. D learning rate is 0.0002 throughout. Each run starts from the same seed and architecture; extra G updates necessarily use extra training noise and compute. The fixed evaluation `z` is identical. Compare A→B for learning-rate effects and B→C for update-balance effects. Predict image readability **and** variety before running B and C. / 학습 전에 B의 G learning rate와 C의 G update 횟수를 직접 입력하세요. A→B에서는 learning rate만, B→C에서는 G update 횟수만 비교하세요.

In [ ]:
#@title Record the plan before training / 변경 전 계획
student_id = '' #@param {type:"string"}
diagnosis = '' #@param {type:"string"}
new_value = None  # TODO B: enter G learning rate 0.0002
g_updates = None  # TODO C: enter G updates per D update 5
prediction = '' #@param {type:"string"}
second_prediction = '' #@param {type:"string"}
success_rule = '' #@param {type:"string"}
record_plan(student_id, diagnosis, new_value, prediction, success_rule, second_prediction, g_updates)

## 3 Complete three useful code blocks / 코드 세 칸
These blocks drive the **intervention** training and generated images. The baseline uses the same supplied reference operations, so the comparison changes only the planned hyperparameter when your code passes its self-check. Small checks verify loss, optimizer changes, gradient path, and output format; they do not reveal your experimental answer. / Baseline은 같은 reference 동작으로 실행됩니다. 코드가 막히면 `check_code(1, use_reference=True)`처럼 전환하세요. 감점은 없고, 보고서에는 계산과 역할을 설명해야 합니다. SyntaxError가 나면 답안 줄을 `None`으로 되돌린 다음 전환하세요.

### Code 1 · D optimizer update
Translate **Week 6 slide 12** (D loss) into one optimizer update. Week 6 slides 8 and 10 give the labels and explain logits. `D` returns logits: use `F.binary_cross_entropy_with_logits` for the two batch terms, then zero gradients, backpropagate and step. Keep the D step from sending gradients into G. Return the scalar D loss. / 강의자료의 D loss 식을 코드로 옮기세요. real/fake target과 `fake.detach()`의 역할을 스스로 결정하고 보고서에 설명하세요.

In [ ]:
def d_update(D, opt, real, fake):
    return None  # TODO: replace this block with working code

check_code(1)

### Code 2 · G optimizer update
Translate **Week 6 slide 14** (non-saturating G loss) into one optimizer update using D logits. Step the G optimizer and return scalar G loss. / G loss와 target을 강의자료에서 확인하세요. D를 통과하는 gradient가 G에 닿아야 합니다.

In [ ]:
def g_update(D, opt, fake):
    return None  # TODO: replace this block with working code

check_code(2)

### Code 3 · Generate fixed-z images
Switch to `eval`, use `torch.no_grad`, map tanh output from [-1,1] to [0,1], return a NumPy array, then restore the prior mode. / BatchNorm이 있는 G의 평가 모드와 밝기 범위를 확인하세요.

In [ ]:
def generate_images(G, z):
    return None  # TODO: replace this block with working code

check_code(3)

In [ ]:
#@title Run the planned comparison / 계획한 비교 실행
run_variant()

## 4 Compare image formation / 생성 이미지 비교
Read the figure from **top to bottom**: real MNIST is reference only; the next three rows show epochs 1, 2, 3. Each epoch has A baseline, B higher G learning rate, C more G updates. Image ID 01 uses the same fixed `z` in every generated panel; it is not paired to real MNIST image 01. Audit the **epoch-3 row** in all three columns. A second figure shows all **64** fixed-z outputs per run; inspect IDs 17–64 for additional repeated or distinct patterns without labeling them. Compare A→B for readability and B→C for readable variety. Repeated recognizable digits suggest restricted variety; noise or ambiguous images alone do not establish mode collapse. `D(real)` and `D(fake)` explain training context but do not measure image quality. / 위에서 아래로 epoch 1→2→3, 왼쪽에서 오른쪽으로 A→B→C입니다. 같은 번호는 같은 고정 `z`입니다. 추가 64장 그리드는 눈으로만 확인하고 label 입력은 처음 16개만 합니다.

Why might concentration arise? G receives feedback on **its generated samples**. If some outputs start to receive useful gradients earlier, updates can move many `z` values toward a subset of patterns. D then adapts; the outcome depends on their balance. A 16-image panel alone cannot establish this cause or full mode coverage. / 이 설명은 가능한 mechanism입니다. 이번 실험이 이를 증명하는 것은 아닙니다.

In [ ]:
#@title Compare generated images / 생성 이미지 비교
show_comparison()

### Visual audit of 16 generated images / 작은 visual audit
For each generated grid, label images **01–16** with one digit 0–9 if clearly recognizable; otherwise `x`. Use the same rule in every run. `readable share` and `distinct readable digits` summarize only this small subjective sample, not full distribution coverage or classifier accuracy. For GAN, the 64-image panel offers a wider visual check; cite IDs 17–64 if they change your conclusion. No additional labels are required. / 모든 run에 같은 기준을 적용하세요. 애매하면 x입니다. 64장 그리드는 보조 관찰용이며 label 48개를 더 입력할 필요는 없습니다.

In [ ]:
#@title Record image IDs 01–16 / 생성 이미지 판정
baseline_labels = '' #@param {type:"string"}
intervention_labels = '' #@param {type:"string"}
second_labels = '' #@param {type:"string"}
save_audit(baseline_labels, intervention_labels, second_labels)

Optional: inspect training curves only to explain observations. G and D losses have different meanings and do not rank image quality. / Loss curve만으로 생성 성능을 판단하지 마세요.

In [ ]:
#@title Optional curves / 선택 확인
show_curves = False #@param {type:"boolean"}
if show_curves:
    optional_loss_plot()

## 5 Q4 One extra setting / 추가 설정 하나
After inspecting A–C, choose **one** extra run. For GAN, use `extra_factor = 'g_lr'`, `extra_value = 0.0004` to extend the G learning-rate comparison (D1:G1), **or** `extra_factor = 'g_updates'`, `extra_value = 3` to test an intermediate D1:G3 at G learning rate 0.0002. Do not change both factors. Write a prediction and a visual decision rule **before** running it. Compare the same fixed-z IDs and label only 01–16 using your earlier rule. You may find no clear trend; do not claim a global optimum from this small experiment. / A–C를 보고 변수 하나를 골라 추가 run 한 번만 실행하세요. 먼저 예측과 판단 기준을 적고, 같은 ID의 이미지로 비교하세요.

In [ ]:
#@title Q4 plan before the extra run / 추가 실험 전 계획
extra_factor = 'g_lr'  # GAN: 'g_lr' or 'g_updates'; VAE: 'beta'
extra_value = None  # TODO: GAN 0.0004 or 3; VAE 5.0 or 0.3
extra_prediction = '' #@param {type:"string"}
extra_decision_rule = '' #@param {type:"string"}
record_extra_plan(extra_factor, extra_value, extra_prediction, extra_decision_rule)

In [ ]:
#@title Run Q4 once / Q4 추가 학습
run_extra()
show_extra()

In [ ]:
#@title Audit Q4 images 01–16 / Q4 이미지 판정
extra_labels = '' #@param {type:"string"}
save_extra_audit(extra_labels)

## 6 Short analysis / 분석 본문
Write **1–2 pages** and then attach the automatic **three-page receipt**. Answer in English or Korean. / 본문 1–2쪽 뒤에 결과 기록 3쪽을 붙여 PDF 하나로 제출하세요.

**Q1 Baseline diagnosis (2 points).** State a visible generation problem, supported by two specific image IDs or features. For VAE, separate prior generation from reconstruction; for GAN, separate unreadable images from repeated readable patterns.

**Q2 Planned changes (2 points).** Explain the two comparisons, your original predictions, what stayed fixed, and one possible cost. Do not rewrite your predictions.

**Q3 What changed (4 points).** For GAN, compare A→B for image readability and B→C for readable variety. Quote image IDs and the three 16-image audits (readable share and distinct digits). Explain how each changed variable could affect G–D training; separate observed evidence from a possible mechanism. A negative or mixed result may receive full marks.

**Q4 Extra setting (2 points).** State your chosen setting and pre-run prediction, compare its image IDs and 16-image audit with the two relevant earlier settings, and identify the best **among tested settings under your stated rule** or explain why no clear ordering appears. State one limit of this small sample; no global optimum or guaranteed mode collapse is expected.

**Code and reproducibility (2 points).** Each block: 0.25 for passing execution + 0.25 for a correct explanation of its training/generation role in your report; 0.5 for a READY receipt consistent with the executed notebook. Reference code is accepted without penalty. A numeric typo is deducted once, in the relevant question only.

## 7 Export receipt / 결과 기록 저장
Write an AI-use statement (`none` if unused). The receipt has settings, audit, metrics, image comparison and a fingerprint. It is an appendix; write your own analysis. Before submission run all cells again, review images and receipt, and save outputs. A cache is reused only while settings and code match. / PDF는 `a2_outputs`에 저장됩니다. Colab 왼쪽 Files에서 받으세요. READY는 자료가 갖춰졌다는 뜻이지 분석의 정답 판정이 아닙니다.

In [ ]:
#@title Export the two-page receipt / 결과 기록 PDF
ai_use = '' #@param {type:"string"}
receipt = make_receipt(ai_use)